**Runtime Parameters**

In [0]:
# Create widgets
dbutils.widgets.text("source_system", "", "Source System")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("source_file_format", "", "Source File Format")

# Get widget values into variables
source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

print(f"source_system: {source_system}")
print(f"table_name: {table_name}")
print(f"landing_timestamp: {landing_timestamp}")
print(f"source_file_format: {source_file_format}")

**Read Landing Data**

In [0]:
landing_path = f"/Volumes/dbw_caresync_dev/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/"

print(f"landing_path: {landing_path}")

In [0]:
import os

# Only a missing path (or missing parent directories) is treated as "no new files".
# PermissionError and any other unexpected exception are NOT caught and will fail the notebook.
try:
    os.stat(landing_path)
except FileNotFoundError:
    message = f"No new files received. Landing path does not exist: {landing_path}"
    print(message)
    dbutils.notebook.exit(message)

In [0]:
if source_file_format == "csv":
    landing_df = spark.read.format("csv").option("header", True).option("inferSchema", True).load(landing_path)
else:
    landing_df = spark.read.format("parquet").load(landing_path)

**Add Ingestion Metadata**

In [0]:
from pyspark.sql.functions import lit, current_timestamp

updated_df = landing_df.withColumn("landing_timestamp", lit(landing_timestamp)) \
                       .withColumn("insert_timestamp", current_timestamp())

**Write to Bronze Delta Table**

In [0]:
updated_df.write.mode("append").saveAsTable(
    f"dbw_caresync_dev.bronze.{table_name}"
)